# Proof of Concept: companies that have rolled back their DEI policies vs. those that didn't

In [5]:
!pip install google-api-python-client isodate
!python -m pip install python-dotenv


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [11]:
import json
import pandas as pd
from googleapiclient.discovery import build
import os
from dotenv import load_dotenv

# =========================
# CONFIG
# =========================
CSV_FILE = "poc_companies.csv"
OUTPUT_FILE = "poc_videos.json"
MAX_RESULTS = 50  # YouTube search API hard caps at 50 per request

# =========================
# SETUP
# =========================
load_dotenv()
API_KEY = os.getenv("YOUTUBE_API_KEY_1")
if not API_KEY:
    raise ValueError("YOUTUBE_API_KEY_1 not found — check your .env file")

youtube = build("youtube", "v3", developerKey=API_KEY)

df = pd.read_csv(CSV_FILE, skipinitialspace=True)
df.columns = df.columns.str.strip()

required_cols = {"company", "CEO", "rollback"}
if not required_cols.issubset(df.columns):
    raise ValueError(f"CSV must contain columns: {required_cols}")

all_videos = []

# =========================
# SEARCH (no keyword filtering here)
# =========================
for _, row in df.iterrows():
    company = row["company"]
    ceo = row["CEO"]
    rollback = row["rollback"]

    if pd.isna(ceo):
        continue

    print(f"Searching for: {ceo} ({company})")

    try:
        search_response = youtube.search().list(
            q=f'{ceo} "interview"',
            part="snippet",
            type="video",
            maxResults=MAX_RESULTS
        ).execute()
    except Exception as e:
        print(f"  Error searching for {ceo}: {e}")
        continue

    items = search_response.get("items", [])
    if not items:
        print(f"  No results for {ceo}")
        continue

    video_ids = [item["id"]["videoId"] for item in items]

    try:
        videos_response = youtube.videos().list(
            part="snippet,contentDetails,statistics",
            id=",".join(video_ids)
        ).execute()
    except Exception as e:
        print(f"  Error fetching video details for {ceo}: {e}")
        continue

    for video in videos_response.get("items", []):
        video_entry = {
            "company": company,
            "CEO": ceo,
            "rollback": rollback,
            "video_id": video["id"],
            "title": video["snippet"]["title"],
            "description": video["snippet"].get("description", ""),
            "channel_title": video["snippet"].get("channelTitle", ""),
            "published_at": video["snippet"].get("publishedAt", ""),
            "duration": video["contentDetails"].get("duration", ""),
            "view_count": video.get("statistics", {}).get("viewCount"),
            "like_count": video.get("statistics", {}).get("likeCount"),
            "comment_count": video.get("statistics", {}).get("commentCount"),
            "url": f"https://www.youtube.com/watch?v={video['id']}",
        }
        all_videos.append(video_entry)

# =========================
# SAVE (unfiltered)
# =========================
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(all_videos, f, indent=2, ensure_ascii=False)

print(f"\nSaved {len(all_videos)} total videos (unfiltered) to {OUTPUT_FILE}")

Searching for: Ron Vachris (Costco)
Searching for: Tim Cook (Apple)
Searching for: Satya Nadella (Microsoft)
Searching for: David Solomon (Goldman Sachs)
Searching for: Chris Kempczinski (McDonald’s)
Searching for: John Furner (Walmart)
Searching for: Andy Jassy (Amazon)
Searching for: Marvin Ellison (Lowe’s)

Saved 380 total videos (unfiltered) to poc_videos.json


In [ ]:
import json
from collections import Counter

# =========================
# CONFIG
# =========================
INPUT_FILE = "poc_videos.json"
FILTERED_OUTPUT_FILE = "poc_videos_filtered.json"

DEI_KEYWORDS = [
    "diversity", "inclusion", "equity", "DEI", "belonging",
    "racial", "racism", "race",
    "minority", "minorities", "representation", "people of color", "poc",
    "black", "hispanic", "asian", "employee", "employees"
    "equality", "gender", "women", "gender pay gap",
    "empowerment", "LGBTQ", "LGBTQIA", "transgender", "pride",
    "sexual orientation", "disability", "accessibility",
    "neurodiversity", "age", "generational", "ageism",
    "workplace", "culture", "employee resource groups",
    "ERG", "affinity groups", "unconscious bias", "microaggression",
    "psychological safety", "ESG", "responsibility",
    "social", "CSR", "stakeholder capitalism",
    "equal opportunity", "affirmative action", "pay equity",
    "diverse hiring", "talent equity", "rollback"
]

# =========================
# LOAD
# =========================
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    all_videos = json.load(f)

print(f"Loaded {len(all_videos)} videos from {INPUT_FILE}")

# =========================
# FILTER (title OR description match)
# =========================
def matches_keywords(video, keywords):
    title = video.get("title", "").lower()
    description = video.get("description", "").lower()
    return any(
        kw.lower() in title or kw.lower() in description
        for kw in keywords
    )

filtered_videos = [
    video for video in all_videos
    if matches_keywords(video, DEI_KEYWORDS)
]

# =========================
# PER-CEO COUNTS
# =========================
total_counts = Counter(v["CEO"] for v in all_videos)
filtered_counts = Counter(v["CEO"] for v in filtered_videos)

print("\nVideos per CEO (filtered / total):")
for ceo in total_counts:
    print(f"  {ceo}: {filtered_counts.get(ceo, 0)} / {total_counts[ceo]}")

# =========================
# SAVE
# =========================
with open(FILTERED_OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(filtered_videos, f, indent=2, ensure_ascii=False)

print(f"\nKept {len(filtered_videos)} / {len(all_videos)} videos matching DEI keywords")
print(f"Saved filtered videos to {FILTERED_OUTPUT_FILE}")

Loaded 380 videos from poc_videos.json

Videos per CEO (filtered / total):
  Ron Vachris: 3 / 30
  Tim Cook: 9 / 50
  Satya Nadella: 9 / 50
  David Solomon: 20 / 50
  Chris Kempczinski: 15 / 50
  John Furner: 7 / 50
  Andy Jassy: 7 / 50
  Marvin Ellison: 10 / 50

Kept 80 / 380 videos matching DEI keywords
Saved filtered videos to poc_videos_filtered.json
